# Joining Land Registry Price Data with Energy Performance Certificates (2015 - 2025)

#### Load Price data

In [122]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)


folder = Path('../data/raw')
file_gen = folder.glob('ppd_data*.csv')
pp_raw = pd.concat([pd.read_csv(f) for f in file_gen], ignore_index=True)
pp_raw['deed_date'] = pp_raw['deed_date'].astype('datetime64[ns]')

#### Load EPC data

In [123]:
epc_raw = pd.read_csv('../data/raw/EDC_OXFORDSHIRE_2015_2025.csv', usecols= [
    'certificate_number', 'address1', 'address2', 'address3', 'postcode',
    'posttown', 'address', 'local_authority', 'local_authority_label',
    'built_form', 'construction_age_band', 'current_energy_efficiency',
    'current_energy_rating', 'energy_consumption_current', 'extension_count', 'floor_height', 'heating_cost_current',
    'inspection_date', 'lodgement_date', 'number_habitable_rooms',
    'number_heated_rooms', 'potential_energy_efficiency', 'property_type',
    'tenure', 'total_floor_area', 'transaction_type', 'uprn', 'uprn_source'
])
epc_raw['inspection_date'] = epc_raw['inspection_date'].astype('datetime64[ns]')
epc_raw['lodgement_date'] = epc_raw['lodgement_date'].astype('datetime64[ns]')

#### Cleaning

Fix trailing whitespace

In [124]:
pp = pp_raw.copy()

for c in pp.select_dtypes(include=['string']).columns:
    pp[c] = pp[c].str.strip()
    print(f"{c} stripped")

unique_id stripped
postcode stripped
property_type stripped
new_build stripped
estate_type stripped
saon stripped
paon stripped
street stripped
locality stripped
town stripped
district stripped
county stripped
transaction_category stripped
linked_data_uri stripped


In [125]:
epc = epc_raw.copy()

for c in epc.select_dtypes(include=['string']).columns:
    epc[c] = epc[c].str.strip()
    print(f"{c} stripped")

certificate_number stripped
address1 stripped
address2 stripped
address3 stripped
postcode stripped
posttown stripped
address stripped
local_authority stripped
local_authority_label stripped
built_form stripped
construction_age_band stripped
current_energy_rating stripped
property_type stripped
tenure stripped
transaction_type stripped
uprn_source stripped


Remove commas

In [126]:
epc_address_cols = ['address1', 'address2', 'address3', 'postcode']
pp_address_cols = ['paon', 'saon', 'street', 'locality', 'postcode']

print(f"Commas replaced in epc:\n{epc[epc_address_cols].apply(lambda col: col.str.count(',')).sum()}")
print(f"Commas replaced in pp:\n{pp[pp_address_cols].apply(lambda col: col.str.count(',')).sum()}")

for c in epc_address_cols:
    epc[c] = epc[c].str.replace(',', '', regex=False)
for c in pp_address_cols:
    pp[c] = pp[c].str.replace(',', '', regex=False)

Commas replaced in epc:
address1    83515.0
address2     3910.0
address3      137.0
postcode        0.0
dtype: float64
Commas replaced in pp:
paon        4356.0
saon          33.0
street         0.0
locality       0.0
postcode       0.0
dtype: float64


Generate address token sets

In [127]:
pp['address'] = pp['paon'].str.cat([pp['saon'], pp['street'], pp['locality'], pp['postcode']], sep=' ', na_rep='')
epc['address'] = epc['address1'].str.cat([epc['address2'], epc['address3'], epc['postcode']], sep=' ', na_rep='').apply(str.upper)

In [128]:
pp['address_tokens'] = pp['address'].str.split().apply(frozenset)
epc['address_tokens'] = epc['address'].str.split().apply(frozenset)

#### Third Join (subset matching)

In [129]:
postcode_match = pp.merge(epc, on='postcode', how='inner')

def compare(row):
    return row['address_tokens_x'].issubset(row['address_tokens_y'])

postcode_match['result'] = postcode_match.apply(compare, axis=1)

In [130]:
epc['uprn'] = epc['uprn'].astype('Int64')

In [131]:
subset_matches = postcode_match[postcode_match['result']]

print(f"Third join (subset method) gives: {subset_matches['unique_id'].nunique()} matches. A matching rate of {100 * subset_matches['unique_id'].nunique()/pp.shape[0]:.1f}%")

Third join (subset method) gives: 105419 matches. A matching rate of 77.8%


Finding records where duplicate words were lost from the address in token setting

In [132]:
multi_match = subset_matches.groupby('unique_id')['uprn'].nunique()
multi_uprn = multi_match[multi_match > 1]

pp['token_count'] = pp['address'].str.count(r"\S+")
lost_tokens = pp[pp['token_count'] > pp['address_tokens'].apply(len)]
result = lost_tokens[lost_tokens['unique_id'].isin(multi_uprn.index)]

Date filtering

In [133]:
third_join = subset_matches[~subset_matches['unique_id'].isin(multi_uprn.index)]
third_join['time_to_sale'] = third_join['deed_date'] - third_join['inspection_date']

drop_neg = third_join[third_join['time_to_sale'] >= '0 days']
third_join_presale_epc = drop_neg.sort_values(by='time_to_sale', ascending=True).drop_duplicates(subset='unique_id')
print(f"Properties remaining with matched pre-sale EPCs: {third_join_presale_epc.shape[0]}")
print(f"Corresponds to a {100 * third_join_presale_epc.shape[0]/pp.shape[0]:.1f}% match rate")

Properties remaining with matched pre-sale EPCs: 91150
Corresponds to a 67.2% match rate


Drop matching and helping columns

In [134]:
cols_to_keep = ['unique_id', 'price_paid', 'deed_date', 'address_x', 'saon', 'paon', 'street', 'locality', 'town', 'district',  
                'postcode', 'property_type_x', 'new_build', 'estate_type',  'transaction_category',  'certificate_number', 
                'built_form', 'construction_age_band', 'current_energy_efficiency', 'current_energy_rating', 
                'energy_consumption_current', 'extension_count', 'floor_height', 'heating_cost_current', 
                'inspection_date', 'lodgement_date', 'number_habitable_rooms', 'number_heated_rooms', 
                'potential_energy_efficiency', 'property_type_y', 'tenure', 'total_floor_area', 
                'transaction_type', 'uprn', 'uprn_source', 'time_to_sale']

df = third_join_presale_epc[cols_to_keep]
df = df.rename(columns={'property_type_x':'property_subtype', 'property_type_y':'property_type', 'time_to_sale':'inspection_to_sale'})


Create EPC age bands from 'inspection_to_sale'

In [135]:
df['inspection_to_sale'] = df['inspection_to_sale'].dt.days

bins = [0, 365, 1095, 1825, 3650, np.inf]
labels = ['0-1 years', '1-3 years', '3-5 years', '5-10 years', '10 years+']

df['EPC_age_band'] = pd.cut(df['inspection_to_sale'], bins=bins, labels=labels, include_lowest=True)

Add flag for expired EPCs (10 years from lodgement_date)

In [136]:
df['EPC_expired'] = ((df['deed_date'] > (df['lodgement_date'] + pd.DateOffset(years=10))))

Adding datetime calendar features to joined/matched data

In [137]:
df['dayofweek'] = df['deed_date'].dt.dayofweek
df['day'] = df['deed_date'].dt.day
df['month'] = df['deed_date'].dt.month
df['quarter'] = df['deed_date'].dt.quarter
df['year'] = df['deed_date'].dt.year

## Missing data

#### Convert invalid values identified in part 1 to null

In [138]:
print(f"Number of records with zero 'number_heated_rooms': {(df['number_heated_rooms'] == 0).sum()}")
print('Converting 0 to NaN...')
df.loc[df['number_heated_rooms'] == 0, 'number_heated_rooms'] = np.nan

print(f"Number of records with zero 'number_heated_rooms': {(df['number_heated_rooms'] == 0).sum()}")

Number of records with zero 'number_heated_rooms': 134
Converting 0 to NaN...
Number of records with zero 'number_heated_rooms': 0


In [139]:
print(f"Number of records with 'floor_height' < 1.5m : {(df['floor_height'] < 1.5).sum()}")
print('Converting to NaN...')
df.loc[df['floor_height'] < 1.5, 'floor_height'] = np.nan

print(f"Number of records with 'floor_height' < 1.5m : {(df['floor_height'] < 1.5).sum()}")

Number of records with 'floor_height' < 1.5m : 1429
Converting to NaN...
Number of records with 'floor_height' < 1.5m : 0


In [140]:
print(f"Number of records with 'property_type' as 'Not Recorded' : {(df['property_type'] == 'Not Recorded').sum()}")
print('Converting to NaN...')
df.loc[df['property_type'] == 'Not Recorded', 'property_type'] = np.nan

print(f"Number of records with 'property_type' as 'Not Recorded' : {(df['property_type'] == 'Not Recorded').sum()}")
df['property_type'].value_counts(dropna=False)

Number of records with 'property_type' as 'Not Recorded' : 0
Converting to NaN...
Number of records with 'property_type' as 'Not Recorded' : 0


property_type
House         74198
Flat          10084
Bungalow       5864
Maisonette      994
NaN               8
Park home         2
Name: count, dtype: int64

In [141]:
df['tenure'].value_counts(dropna=False)

tenure
owner-occupied      63106
unknown             21458
rented (private)     6174
rented (social)       412
Name: count, dtype: int64

In [142]:
print(f"Number of records with 'tenure' as 'unknown' : {(df['tenure'] == 'unknown').sum()}")
print('Converting to NaN...')
df.loc[df['tenure'] == 'unknown', 'tenure'] = np.nan

print(f"Number of records with 'tenure' as 'unknown' : {(df['tenure'] == 'unknown').sum()}")

Number of records with 'tenure' as 'unknown' : 21458
Converting to NaN...
Number of records with 'tenure' as 'unknown' : 0


#### Measure missingness

In [143]:

fields_miss = df.isna().mean().drop_duplicates().reset_index()

fields_miss.columns = ['field_name', 'missing']
fields_miss['missing'] = round(fields_miss['missing'] * 100, 2)
fields_miss = fields_miss.set_index(keys='field_name')
print(fields_miss)

                     missing
field_name                  
unique_id               0.00
saon                   93.52
street                  1.89
locality               59.61
built_form              2.20
extension_count        32.33
floor_height            2.25
number_heated_rooms    32.48
property_type           0.01
tenure                 23.54
transaction_type        0.02
uprn                    0.19


#### Apply part 1 exclusions

- *Proposed: Exclude these 7 records with multiple rooms and < 9 sqm. Exclude the record with 3 sqm. Flag the 19 records that are 1 room flats (7-9 sqm) and check sale prices after join before deciding.*
- *Proposed: Exclude the 7 records with >2000 sqm and <= 7 rooms (8206-7322-4290-4245-7906, 0552-3041-5201-3644-6204, 1032-5920-2209-0815-0202, 0044-2876-6518-9608-3961, 8171-6324-5580-7887-6996, 3434-1132-3000-0328-7226, 5135-0635-7000-0689-9206)*
- Two have energy_consumption_current >200 and are Maisonette/Flat which flags them as likely errors. *Proposed: Exclude 2051-7735-3040-1307-9975 and 0061-1212-2504-8687-0900*
-  `construction_age_band` exclude 201 as an error

Checking for propertes with <= 9 sqm total_floor_area.

In [144]:
print(df[df['total_floor_area'] <= 9])
print(df.shape)
df = df[~(df['total_floor_area'] <= 9)]
print(df.shape)

                                    unique_id  price_paid  deed_date                 address_x saon paon      street locality    town          district  postcode property_subtype new_build estate_type transaction_category        certificate_number   built_form         construction_age_band  current_energy_efficiency current_energy_rating  energy_consumption_current  extension_count  floor_height  heating_cost_current inspection_date lodgement_date  number_habitable_rooms  number_heated_rooms  potential_energy_efficiency property_type          tenure  total_floor_area transaction_type          uprn      uprn_source  inspection_to_sale EPC_age_band  EPC_expired  dayofweek  day  month  quarter  year
2584364  44F406B6-C504-1095-E063-4704A8C048D4      107250 2025-10-24  24  MANOR ROAD  OX28 3SR  NaN   24  MANOR ROAD      NaN  WITNEY  WEST OXFORDSHIRE  OX28 3SR                F         N           L                    A  0360-2874-6282-9391-2035  End-Terrace  England and Wales: 1983-1990    

- Dropped 1 property with 10 sqm floor area and 2 rooms

Checking for the 7 high floor area records flagged for exclusion (>2000 sqm and <= 7 rooms)

In [145]:
print(df[df['certificate_number'].isin(['8206-7322-4290-4245-7906', 
                               '0552-3041-5201-3644-6204', 
                               '1032-5920-2209-0815-0202', 
                               '0044-2876-6518-9608-3961', 
                               '8171-6324-5580-7887-6996', 
                               '3434-1132-3000-0328-7226', 
                               '5135-0635-7000-0689-9206'])])

df = df[~df['certificate_number'].isin(['8206-7322-4290-4245-7906', 
                               '0552-3041-5201-3644-6204', 
                               '1032-5920-2209-0815-0202', 
                               '0044-2876-6518-9608-3961', 
                               '8171-6324-5580-7887-6996', 
                               '3434-1132-3000-0328-7226', 
                               '5135-0635-7000-0689-9206'])]

print(df.shape)

                                    unique_id  price_paid  deed_date                           address_x saon             paon         street    locality              town           district postcode property_subtype new_build estate_type transaction_category        certificate_number     built_form           construction_age_band  current_energy_efficiency current_energy_rating  energy_consumption_current  extension_count  floor_height  heating_cost_current inspection_date lodgement_date  number_habitable_rooms  number_heated_rooms  potential_energy_efficiency property_type          tenure  total_floor_area transaction_type          uprn      uprn_source  inspection_to_sale EPC_age_band  EPC_expired  dayofweek  day  month  quarter  year
1715797  34428D7E-6659-B86C-E050-A8C06205059C      926100 2016-05-13  THE OLD LAUNDRY   HARPSDEN RG9 4HL  NaN  THE OLD LAUNDRY            NaN    HARPSDEN  HENLEY-ON-THAMES  SOUTH OXFORDSHIRE  RG9 4HL                D         N           F              

- 3 in dataset, excluded

Two records have current_energy_efficiency > 110, energy_consumption_current >200 and are Maisonette/Flat which flags them as likely errors (2051-7735-3040-1307-9975 and 0061-1212-2504-8687-0900)

In [146]:
print(df[df['certificate_number'].isin(['2051-7735-3040-1307-9975', 
                               '0061-1212-2504-8687-0900'])])

Empty DataFrame
Columns: [unique_id, price_paid, deed_date, address_x, saon, paon, street, locality, town, district, postcode, property_subtype, new_build, estate_type, transaction_category, certificate_number, built_form, construction_age_band, current_energy_efficiency, current_energy_rating, energy_consumption_current, extension_count, floor_height, heating_cost_current, inspection_date, lodgement_date, number_habitable_rooms, number_heated_rooms, potential_energy_efficiency, property_type, tenure, total_floor_area, transaction_type, uprn, uprn_source, inspection_to_sale, EPC_age_band, EPC_expired, dayofweek, day, month, quarter, year]
Index: []


- Neither record in set

`construction_age_band` exclude 201 as an error

In [147]:
print(df.loc[df['construction_age_band'].apply(len) < 4, 'construction_age_band'])

df = df[df['construction_age_band'].apply(len) >= 4]

df.shape

2055929    201
2055958    201
2055900    201
Name: construction_age_band, dtype: str


(91143, 43)

- Excluded 3 records with 'construction_age_band' = '201'

#### Addressing columns with missing values

New dwellings are systematically missing room counts from initial EPCs. Can they be filled from subsequent records?

In [148]:
print(((df['transaction_type'] == 'New dwelling') & (df['number_heated_rooms'].isna())).sum())
nd_nanroom_uprn = df['uprn'][((df['transaction_type'] == 'New dwelling') & (df['number_heated_rooms'].isna()))].values

epc_raw = pd.read_csv('../data/raw/EDC_OXFORDSHIRE_2015_2025.csv', usecols= [
    'certificate_number', 'address1', 'address2', 'address3', 'postcode',
    'posttown', 'address', 'local_authority', 'local_authority_label',
    'built_form', 'construction_age_band', 'current_energy_efficiency',
    'current_energy_rating', 'energy_consumption_current', 'extension_count', 'floor_height', 'heating_cost_current',
    'inspection_date', 'lodgement_date', 'number_habitable_rooms',
    'number_heated_rooms', 'potential_energy_efficiency', 'property_type',
    'tenure', 'total_floor_area', 'transaction_type', 'uprn', 'uprn_source'
])
epc_raw['inspection_date'] = epc_raw['inspection_date'].astype('datetime64[ns]')
epc_raw['lodgement_date'] = epc_raw['lodgement_date'].astype('datetime64[ns]')




27473


In [149]:
nd_nanroom = epc_raw[epc_raw['uprn'].isin(nd_nanroom_uprn)]

nd_nanroom.groupby('uprn')['number_heated_rooms'].count() > 0


uprn
1.000400e+10     True
1.000400e+10    False
1.000400e+10    False
1.000987e+10     True
1.001187e+10    False
                ...  
2.000017e+11     True
2.000018e+11     True
2.000024e+11    False
2.000029e+11    False
2.000047e+11    False
Name: number_heated_rooms, Length: 22344, dtype: bool